In [17]:
import pandas as pd

In [18]:
df = pd.read_csv('votacoes_full_with_communities_complete.csv')
df.head(5)

,id,data,idOrgao,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros,descricao,ano,...,orientacao_GOV,tipoAutor,idDeputadoAutor,nomeAutor,data_ref,legislatura,author_prev_community,prev_community_0_size,prev_community_1_size,prev_community_2_size
0,103255-3,2003-02-19,180,PLEN,1.0,0,0,0,Aprovado o Requerimento,2003,...,Neutro,Deputado(a),74218.0,Walter Pinheiro,1,52,NaN,NaN,NaN,0
1,102277-6,2003-02-19,180,PLEN,1.0,0,0,0,Aprovado requerimento n. 264/2002 de Líderes q...,2003,...,Neutro,Deputado(a),74218.0,Walter Pinheiro,1,52,NaN,NaN,NaN,0
2,105029-15,2003-03-18,180,PLEN,1.0,0,0,0,Aprovado REQ 450/2003 do Sr. Maurício Rabelo q...,2003,...,Neutro,Deputado(a),73947.0,Maurício Rabelo,8,52,NaN,NaN,NaN,0
3,107184-2,2003-03-18,180,PLEN,1.0,0,0,0,Aprovado o Requerimento,2003,...,Neutro,Deputado(a),73947.0,Maurício Rabelo,8,52,NaN,NaN,NaN,0
4,19701-51,2003-03-20,180,PLEN,1.0,0,0,0,"Aprovado o Projeto de Lei nº 3.462, de 2000, r...",2003,...,Neutro,Deputado(a),74082.0,Paulo Rocha,10,52,NaN,NaN,NaN,0


In [19]:
df = df[df['legislatura'] != 52]
df['majority'] = (
    ((df['author_prev_community'] == 0) & (df['prev_community_0_size'] > df['prev_community_1_size'])) |
    ((df['author_prev_community'] == 1) & (df['prev_community_1_size'] > df['prev_community_0_size']))
)
df.head(5)
df.head(5)

,id,data,idOrgao,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros,descricao,ano,...,tipoAutor,idDeputadoAutor,nomeAutor,data_ref,legislatura,author_prev_community,prev_community_0_size,prev_community_1_size,prev_community_2_size,majority
5644,340896-3,2007-02-14,180,PLEN,1.0,0,0,0,Aprovada a Requerimento de Urgência (Art. 155 ...,2007,...,Deputado(a),74399.0,Onyx Lorenzoni,333,53,1.0,330.0,252.0,0,False
5645,340896-3,2007-02-14,180,PLEN,1.0,0,0,0,Aprovada a Requerimento de Urgência (Art. 155 ...,2007,...,Deputado(a),74399.0,Onyx Lorenzoni,333,53,1.0,330.0,252.0,0,False
5646,340368-33,2007-02-15,180,PLEN,1.0,0,0,0,Aprovada a Redação Final.,2007,...,Deputado(a),73666.0,Jovair Arantes,334,53,0.0,330.0,252.0,0,True
5647,340368-33,2007-02-15,180,PLEN,1.0,0,0,0,Aprovada a Redação Final.,2007,...,Deputado(a),73431.0,Antonio Carlos Pannunzio,334,53,1.0,330.0,252.0,0,False
5648,340368-33,2007-02-15,180,PLEN,1.0,0,0,0,Aprovada a Redação Final.,2007,...,Deputado(a),73982.0,Luciano Castro,334,53,0.0,330.0,252.0,0,True


In [20]:
print(df['aprovacao'].value_counts())


aprovacao
1.0    50059
0.0    19918
Name: count, dtype: int64


In [21]:
print(df['majority'].value_counts())


majority
False    50095
True     23075
Name: count, dtype: int64


In [23]:
counts = df.groupby('aprovacao')['majority'].value_counts().unstack(fill_value=0)
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

print("Counts of True/False in 'majority' for each 'aprovacao':")
print(counts)
print("\nPercentages of True/False in 'majority' for each 'aprovacao':")
print(percentages.round(2))



Counts of True/False in 'majority' for each 'aprovacao':
majority   False  True 
aprovacao              
0.0        12971   6947
1.0        34955  15104

Percentages of True/False in 'majority' for each 'aprovacao':
majority   False  True 
aprovacao              
0.0        65.12  34.88
1.0        69.83  30.17


In [24]:
df_votosSim = df[df['votosSim'] != 0]
counts_votosSim = df_votosSim.groupby('aprovacao')['majority'].value_counts().unstack(fill_value=0)
percentages_votosSim = counts_votosSim.div(counts_votosSim.sum(axis=1), axis=0) * 100

print("Counts of True/False in 'majority' for each 'aprovacao' (where votosSim != 0):")
print(counts_votosSim)
print("\nPercentages of True/False in 'majority' for each 'aprovacao' (where votosSim != 0):")
print(percentages_votosSim.round(2))


Counts of True/False in 'majority' for each 'aprovacao' (where votosSim != 0):
majority   False  True 
aprovacao              
0.0         7025   3564
1.0         8007   4789

Percentages of True/False in 'majority' for each 'aprovacao' (where votosSim != 0):
majority   False  True 
aprovacao              
0.0        66.34  33.66
1.0        62.57  37.43


In [25]:
# Filter rows where both votosSim and votosNao are different from 0
df_similar_votes = df[(df['votosSim'] != 0) & (df['votosNao'] != 0)]

# Calculate the ratio between votosSim and votosNao
# We want votosSim to be between 40% and 60% of the sum of votosSim + votosNao
total_votes = df_similar_votes['votosSim'] + df_similar_votes['votosNao']
votosSim_ratio = df_similar_votes['votosSim'] / total_votes

# Keep only rows where votosSim is between 40% and 60% of the total (i.e., similar to votosNao)
df_similar_votes = df_similar_votes[(votosSim_ratio >= 0.4) & (votosSim_ratio <= 0.6)]

# Group and count as before
counts_similar = df_similar_votes.groupby('aprovacao')['majority'].value_counts().unstack(fill_value=0)
percentages_similar = counts_similar.div(counts_similar.sum(axis=1), axis=0) * 100

print("Counts of True/False in 'majority' for each 'aprovacao' (where votosSim and votosNao are both nonzero and similar):")
print(counts_similar)
print("\nPercentages of True/False in 'majority' for each 'aprovacao' (where votosSim and votosNao are both nonzero and similar):")
print(percentages_similar.round(2))


Counts of True/False in 'majority' for each 'aprovacao' (where votosSim and votosNao are both nonzero and similar):
majority   False  True 
aprovacao              
0.0          962    292
1.0          191     56

Percentages of True/False in 'majority' for each 'aprovacao' (where votosSim and votosNao are both nonzero and similar):
majority   False  True 
aprovacao              
0.0        76.71  23.29
1.0        77.33  22.67
